# 05. 量子化の方式・ビット数を比べる

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 05 です。

**同じモデル**を、いろいろな方式・ビット数で量子化して、「どれだけ小さくなるか」「どれだけ賢さが落ちるか」「速さはどうか」を並べて比べます。

使うモデル: [`Qwen/Qwen3-8B`](https://huggingface.co/Qwen/Qwen3-8B)（8.2B、Apache-2.0）

- 比べる基準として、**量子化しない bf16（16bit）も L4 に載る**大きさが必要 → 8B 級を選んだ（31B の bf16 は 62GB で載らない）
- エンジンはすべて `transformers` にそろえる（エンジンの差ではなく、量子化の差だけを見るため）

| # | 方式 | ビット数 | 特徴 |
|---|---|---|---|
| 1 | bf16 | 16 | 量子化なし。基準 |
| 2 | bitsandbytes 8bit（LLM.int8） | 8 | ほぼ劣化しないと言われる |
| 3 | bitsandbytes 4bit NF4 | 4 | 02 / 03 で使った方式 |
| 4 | bitsandbytes 4bit FP4 | 4 | NF4 の兄弟。数の並べ方が違う |
| 5 | HQQ 4bit | 4 | 準備データなしで、読み込み時にすばやく量子化 |
| 6 | HQQ 3bit | 3 | |
| 7 | HQQ 2bit | 2 | ここまで下げるとどうなるか |

測るもの:

1. **VRAM**（モデルの大きさ）
2. **パープレキシティ（PPL）** … 文章の続きをどれだけ正しく予想できるか。**小さいほど良い**。英語（WikiText-2）と日本語（このリポジトリの解説ページ）の2つ
3. **10問の正答数** … 03 / 04 と同じ5問 + 少し難しい5問（thinking オフ）
4. **速さ**（1件ずつ、128 トークン）

「想定どおり」とは:

- ビット数を下げるほど VRAM が減る
- 8bit は bf16 とほぼ同じ（PPL の悪化 3% 以内）
- 4bit は実用範囲（PPL の悪化 10% 以内、正答数の低下 1 問以内）
- 2bit は大きく崩れる（PPL が最も悪い）

所要時間の目安: 30〜45 分。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU を確認して、ライブラリを入れる

In [ ]:
# ノート本体では torch を使わない（各構成は別プロセスで動かし、GPU メモリを毎回まっさらにするため）
import subprocess
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
gpu_name, mem_mib = [x.strip() for x in q.strip().split(",")]
vram_total_gb = int(mem_mib) / 1024
assert "L4" in gpu_name, f"GPU が L4 ではありません: {gpu_name}"
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB")

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes hqq datasets 2>&1 | tail -2
!python -c "import torch, transformers, bitsandbytes, hqq; print('torch', torch.__version__, '/ transformers', transformers.__version__, '/ bitsandbytes', bitsandbytes.__version__, '/ hqq', hqq.__version__)"

## 2. 評価用のデータを用意する

- 英語: WikiText-2（テスト用の文章。PPL の計測でよく使われる定番）
- 日本語: このリポジトリの解説ページ（docs/01〜06）

どちらも先頭から 1,024 トークンずつ、最大 8 区切り分を使います。

In [ ]:
import json, urllib.request
from datasets import load_dataset

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
en_text = "\n".join(t for t in wiki["text"] if t.strip())

base = "https://raw.githubusercontent.com/moruku36/colab-oss-lab/main/docs/"
pages = ["01-what-is-colab.md", "02-google-ai-pro.md", "03-what-you-can-do.md",
         "04-oss-models.md", "05-gpu-basics.md", "06-quantization.md"]
ja_text = "\n\n".join(urllib.request.urlopen(base + p).read().decode("utf-8").split("\n---\n\n## 関連する実験")[0]
                        for p in pages)
json.dump({"en": en_text[:200000], "ja": ja_text}, open("/content/eval_text.json", "w"), ensure_ascii=False)
print("英語:", len(en_text), "文字 / 日本語:", len(ja_text), "文字")

## 3. 計測用のスクリプトを書く

構成ごとに**別プロセス**で動かします（前の構成の GPU メモリが残らないように）。

HQQ は、transformers 5.x の `HqqConfig` が「まだ使えない」エラーになるため、`hqq` ライブラリで Linear 層を直接置き換えています。

In [ ]:
%%writefile qbench.py
import argparse, json, math, re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

p = argparse.ArgumentParser()
p.add_argument("--name", required=True)
p.add_argument("--method", required=True, choices=["bf16", "bnb8", "nf4", "fp4", "hqq"])
p.add_argument("--bits", type=int, default=4)
p.add_argument("--group", type=int, default=64)
a = p.parse_args()

MODEL_ID = "Qwen/Qwen3-8B"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"

kw = dict(dtype=torch.bfloat16, device_map={"": 0})
if a.method == "bnb8":
    kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
elif a.method in ("nf4", "fp4"):
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type=a.method,
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
if a.method == "hqq":
    # transformers 5.x の HqqConfig は「まだ使えない」と出るので、hqq ライブラリで直接置き換える:
    # bf16 で読み込んだあと、lm_head 以外の Linear を1つずつ HQQ の量子化レイヤーに差し替える
    from hqq.core.quantize import BaseQuantizeConfig, HQQLinear
    qcfg = BaseQuantizeConfig(nbits=a.bits, group_size=a.group)
    def swap(mod):
        for n, ch in mod.named_children():
            if isinstance(ch, torch.nn.Linear) and n != "lm_head":
                setattr(mod, n, HQQLinear(ch, qcfg, compute_dtype=torch.bfloat16, device="cuda:0", del_orig=True))
            else:
                swap(ch)
    swap(model)
    torch.cuda.empty_cache()
model.eval()
load_sec = time.time() - t0
torch.cuda.empty_cache()
vram_load = torch.cuda.memory_allocated() / 1024**3  # 方式によらず同じ物差しで測る
footprint = model.get_memory_footprint() / 1024**3

# 1) パープレキシティ（1,024 トークンずつ、最大 8 区切り）
texts = json.load(open("/content/eval_text.json"))
@torch.no_grad()
def ppl(text, win=1024, n=8):
    ids = tok(text, return_tensors="pt").input_ids[0]
    losses = []
    for i in range(min(n, len(ids) // win)):
        x = ids[i * win:(i + 1) * win].unsqueeze(0).to(0)
        losses.append(model(x, labels=x).loss.float().item())
    return math.exp(sum(losses) / len(losses)), len(losses) * win
ppl_en, n_en = ppl(texts["en"])
ppl_ja, n_ja = ppl(texts["ja"])

# 2) 10問（まとめて生成、thinking オフ）
SYSTEM = "You are a helpful assistant. Answer in Japanese. 最後の行に必ず「答え: <数字>」の形で答えだけを書いてください。"
QUIZ = [
    ("ある数に3を足して2倍すると、その数の3倍より4小さくなります。ある数はいくつですか。", 10),
    ("1から100までの整数のうち、3でも5でも割り切れないものはいくつありますか。", 53),
    ("英単語 strawberry の中に、アルファベットの r は何個含まれていますか。", 3),
    ("A、B、C、D の4人が横一列に並びます。AとBが隣り合わない並び方は何通りですか。", 12),
    ("時計が3時15分を指しているとき、長針と短針がつくる小さいほうの角は何度ですか。", 7.5),
    ("7で割ると3余り、5で割ると2余る2桁の自然数のうち、いちばん小さいものは何ですか。", 17),
    ("定価の2割引きで買った品物の代金が960円でした。定価は何円ですか。", 1200),
    ("1から50までの整数をすべて足すといくつですか。", 1275),
    ("サイコロを2個振ったとき、出た目の和が7になる出方は、36通りのうち何通りですか。", 6),
    ("A地点からB地点まで、時速4kmで歩くと時速12kmの自転車より1時間遅く着きます。AB間の距離は何kmですか。", 6),
]
def extract(text):
    text = text.replace(",", "").replace("，", "")
    m = re.findall(r"答え\s*[:：]\s*\**\s*([0-9]+(?:\.[0-9]+)?)", text) or re.findall(r"([0-9]+(?:\.[0-9]+)?)", text)
    return float(m[-1]) if m else None
prompts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False) for q, _ in QUIZ]
enc = tok(prompts, return_tensors="pt", padding=True).to(0)
t = time.time()
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=384, do_sample=False)
quiz_sec = time.time() - t
quiz = []
for (q, ans), o in zip(QUIZ, out):
    text = tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    got = extract(text)
    quiz.append(dict(q=q, expected=ans, got=got, correct=got is not None and abs(got - ans) < 1e-6, text=text))

# 3) 速さ（1件ずつ、ちょうど 128 トークン出させる）
sample_q = "小学校の児童にも分かる言葉で、GPUとVRAMの違いを3文で説明してください。"
s_prompt = tok.apply_chat_template([{"role": "user", "content": sample_q}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)
s_enc = tok([s_prompt], return_tensors="pt").to(0)
with torch.no_grad():
    model.generate(**s_enc, max_new_tokens=8, do_sample=False)  # 準備運動
    torch.cuda.synchronize(); t = time.time()
    s_out = model.generate(**s_enc, max_new_tokens=128, min_new_tokens=128, do_sample=False)
    torch.cuda.synchronize(); gen_sec = time.time() - t
with torch.no_grad():
    a_out = model.generate(**s_enc, max_new_tokens=200, do_sample=False)
sample = re.sub(r"<think>.*?</think>", "", tok.decode(a_out[0][s_enc.input_ids.shape[1]:], skip_special_tokens=True), flags=re.S).strip()

res = dict(name=a.name, method=a.method, bits=(16 if a.method == "bf16" else 8 if a.method == "bnb8" else 4 if a.method in ("nf4", "fp4") else a.bits),
           group=a.group if a.method == "hqq" else None,
           load_sec=load_sec, vram_load_gb=vram_load, footprint_gb=footprint,
           vram_peak_gb=torch.cuda.max_memory_allocated() / 1024**3,
           ppl_en=ppl_en, ppl_ja=ppl_ja, ppl_tokens=dict(en=n_en, ja=n_ja),
           quiz=quiz, quiz_correct=sum(r["correct"] for r in quiz), quiz_sec=quiz_sec,
           tps=128 / gen_sec, sample=sample)
json.dump(res, open(f"/content/q_{a.name}.json", "w"), ensure_ascii=False, indent=1)
print(f"OK {a.name}: VRAM {footprint:.2f}GB / PPL en {ppl_en:.2f} ja {ppl_ja:.2f} / 正答 {res['quiz_correct']}/10 / {res['tps']:.1f} tok/s")

## 4. 7つの構成を順に計測する

In [ ]:
import json, os, re, subprocess, time

CONFIGS = [
    ("bf16",  "bf16（量子化なし）",    ["--method", "bf16"]),
    ("bnb8",  "bitsandbytes 8bit",     ["--method", "bnb8"]),
    ("nf4",   "bitsandbytes 4bit NF4", ["--method", "nf4"]),
    ("fp4",   "bitsandbytes 4bit FP4", ["--method", "fp4"]),
    ("hqq4",  "HQQ 4bit",              ["--method", "hqq", "--bits", "4"]),
    ("hqq3",  "HQQ 3bit",              ["--method", "hqq", "--bits", "3"]),
    ("hqq2",  "HQQ 2bit",              ["--method", "hqq", "--bits", "2"]),
]
results = {}
for name, label, args in CONFIGS:
    if os.path.exists(f"/content/q_{name}.json"):  # 前に成功した構成は測り直さない
        r = json.load(open(f"/content/q_{name}.json")); r["label"] = label
        results[name] = r
        print(f"[{label}] 前回の結果を使う")
        continue
    t = time.time()
    p = subprocess.run(["python", "qbench.py", "--name", name] + args, capture_output=True, text=True)
    log = p.stdout + p.stderr
    open(f"/content/q_{name}.log", "w").write(log)
    if p.returncode == 0 and os.path.exists(f"/content/q_{name}.json"):
        r = json.load(open(f"/content/q_{name}.json")); r["label"] = label
        results[name] = r
        print(f"[{label}] {(time.time() - t) / 60:.1f} 分 → " + log.strip().splitlines()[-1])
    else:
        lines = log.splitlines()
        causes = [l for l in lines if re.search(r"(Error|error:|out of memory)", l)]
        print(f"[{label}] 失敗（{(time.time() - t) / 60:.1f} 分）: " + (causes[-1][:300] if causes else "ログを確認: /content/q_" + name + ".log"))

## 5. まとめて、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta
import torch, transformers, bitsandbytes, hqq  # バージョン表示のためだけ（GPU は使わない）

b = results.get("bf16")
def d(x, base):
    return f"{(x / base - 1) * 100:+.1f}%" if base else "-"

rows = []
for name, label, _ in CONFIGS:
    r = results.get(name)
    if r is None:
        rows.append(f"| {label} | 失敗 | - | - | - | - | - |")
        continue
    rows.append(f"| {label} | {r['bits']} | {r['vram_load_gb']:.2f} GB | "
                f"{r['ppl_en']:.2f}（{d(r['ppl_en'], b and b['ppl_en'])}） | "
                f"{r['ppl_ja']:.2f}（{d(r['ppl_ja'], b and b['ppl_ja'])}） | "
                f"{r['quiz_correct']} / 10 | {r['tps']:.1f} |")

def get(n, k): return results[n][k] if n in results else None
checks = {}
order = [n for n in ["bf16", "bnb8", "nf4", "hqq3", "hqq2"] if n in results]
checks["ビット数を下げるほど VRAM が減る（bf16 > 8bit > 4bit > 3bit > 2bit）"] = (
    len(order) == 5 and all(results[order[i]]["vram_load_gb"] > results[order[i + 1]]["vram_load_gb"] for i in range(4)))
checks["8bit の PPL 悪化が 3% 以内（英・日）"] = bool(b and "bnb8" in results) and all(
    get("bnb8", k) / b[k] - 1 <= 0.03 for k in ("ppl_en", "ppl_ja"))
four = [n for n in ("nf4", "fp4", "hqq4") if n in results]
checks["4bit（NF4 / FP4 / HQQ）の PPL 悪化が 10% 以内、正答の低下が 1 問以内"] = bool(b and len(four) == 3) and all(
    results[n][k] / b[k] - 1 <= 0.10 for n in four for k in ("ppl_en", "ppl_ja")) and all(
    results[n]["quiz_correct"] >= b["quiz_correct"] - 1 for n in four)
checks["2bit の PPL がいちばん悪い"] = "hqq2" in results and all(
    results["hqq2"]["ppl_en"] >= r["ppl_en"] for r in results.values())
ok = all(checks.values())

now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
L = [
    "# 実行記録: 05 量子化の方式・ビット数を比べる", "",
    f"- 実行日: {now}", "- 実行場所: Google Colab",
    f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
    "- モデル: Qwen/Qwen3-8B",
    f"- torch {torch.__version__} / transformers {transformers.__version__} / bitsandbytes {bitsandbytes.__version__} / hqq {hqq.__version__}",
    "- PPL: 1,024 トークン × 最大 8 区切り（英語 WikiText-2 test / 日本語 docs/01〜06）",
    "- 10問: greedy、thinking オフ、まとめて生成 / 速さ: 1件ずつ 128 トークン",
    "- HQQ の group size: 64",
    f"- 想定どおりか: {'はい' if ok else 'いいえ'}", "",
    "## まとめ", "",
    "| 構成 | ビット | VRAM（読み込み後） | PPL 英語（bf16 比） | PPL 日本語（bf16 比） | 10問の正答 | 速さ（トークン/秒） |",
    "|---|---|---|---|---|---|---|",
] + rows + ["", "## 判定", ""] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + [""]
L += ["## 問題ごとの正誤", "", "| # | 正解 | " + " | ".join(results[n]["label"] for n, _, _ in CONFIGS if n in results) + " |",
      "|---|---|" + "---|" * len(results)]
for i, (q, ans) in enumerate([(x["q"], x["expected"]) for x in next(iter(results.values()))["quiz"]], 1):
    cells = []
    for n, _, _ in CONFIGS:
        if n in results:
            x = results[n]["quiz"][i - 1]
            cells.append(("○" if x["correct"] else "×") + f" {x['got']}")
    L.append(f"| {i} | {ans} | " + " | ".join(cells) + " |")
L += ["", "## 返事の例（GPUとVRAMの違いを3文で）", ""]
for n, label, _ in CONFIGS:
    if n in results:
        L += [f"### {label}", "", "```", results[n]["sample"][:600], "```", ""]
print("\n".join(L))
json.dump(results, open("/content/q_all.json", "w"), ensure_ascii=False, indent=1)

## 終わったら

**ランタイム → セッションを管理 → 解放** を必ず押してください。